# DNABERT-2 Finetuning for Promoter Classification

This tutorial shows an **end-to-end DNABERT-2 finetuning workflow** for binary promoter classification using Hugging Face Transformers.

> Notes
> - This notebook is designed as an example tutorial and may need dependency installation in a clean environment.
> - If you run on a GPU machine, training will be much faster.

## 1) Install dependencies

Uncomment and run the next cell if your environment does not already have these packages.

In [ ]:
# %pip install -q transformers datasets evaluate scikit-learn accelerate pandas numpy torch

## 2) Imports and configuration

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)

set_seed(42)
MODEL_NAME = "zhihan1996/DNABERT-2-117M"
MAX_LENGTH = 256

## 3) Load promoter classification data

This example expects a CSV with:
- `sequence`: DNA string (`A/C/G/T/N`)
- `label`: class index (0 or 1)

A tiny synthetic fallback dataset is provided so the notebook can run out-of-the-box.

In [ ]:
data_path = Path("data/data_DNABERT/promoter_classification.csv")

if data_path.exists():
    df = pd.read_csv(data_path)
else:
    # Minimal synthetic dataset fallback
    df = pd.DataFrame(
        {
            "sequence": [
                "TTGACATATAGCTCAGTCCTAGGTATAATGCTAGC",
                "CGTATCGATCGATCGATCGATCGATCGATCGATCG",
                "TTGACAAGGCTATAATGCGGCGTATATATCGCGCG",
                "GCGCGCGCGCGCGTTTTAAAACCCCGGGGTTTTAAA",
                "TTGACAACACACATATAATGGGCGCGCGATATATAT",
                "ATATATATATATATATGCGCGCGCGCGCGCGCGCGC",
            ],
            "label": [1, 0, 1, 0, 1, 0],
        }
    )

required_cols = {"sequence", "label"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df[["sequence", "label"]].dropna()
df["sequence"] = df["sequence"].str.upper().str.replace(r"[^ACGTN]", "N", regex=True)
df["label"] = df["label"].astype(int)

df.head()

## 4) Train/validation/test split

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    random_state=42,
    stratify=df["label"] if df["label"].nunique() > 1 else None,
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["label"] if temp_df["label"].nunique() > 1 else None,
)

dataset = DatasetDict(
    {
        "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
        "validation": Dataset.from_pandas(val_df.reset_index(drop=True)),
        "test": Dataset.from_pandas(test_df.reset_index(drop=True)),
    }
)

dataset

## 5) Tokenization with DNABERT-2 tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

def tokenize_batch(batch):
    return tokenizer(
        batch["sequence"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized = dataset.map(tokenize_batch, batched=True)
tokenized = tokenized.remove_columns([col for col in tokenized["train"].column_names if col not in {"input_ids", "attention_mask", "label"}])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized

## 6) Build model and metrics

In [ ]:
num_labels = int(df["label"].nunique())
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=max(num_labels, 2),
    trust_remote_code=True,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="binary" if num_labels == 2 else "macro"),
        "precision": precision_score(labels, preds, average="binary" if num_labels == 2 else "macro", zero_division=0),
        "recall": recall_score(labels, preds, average="binary" if num_labels == 2 else "macro", zero_division=0),
    }

## 7) Finetune DNABERT-2

In [ ]:
training_args = TrainingArguments(
    output_dir="./outputs/dnabert2-promoter",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

## 8) Evaluate on test set

In [ ]:
test_metrics = trainer.evaluate(tokenized["test"])
test_metrics

## 9) Run inference on new sequences

In [ ]:
new_sequences = [
    "TTGACAGGTTATAATCCGCGTATATGCGCGATAT",
    "GCGCGCGCGCGATATATATATATATATATATATA",
]

inputs = tokenizer(new_sequences, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    logits = model(**inputs).logits

probs = torch.softmax(logits, dim=-1).cpu().numpy()
preds = probs.argmax(axis=-1)

for seq, pred, prob in zip(new_sequences, preds, probs):
    print({"sequence": seq, "predicted_label": int(pred), "probabilities": prob.tolist()})